In [4]:
import os
import shutil
import json
import re

def read_txt_file(file_path):
    with open(file_path, 'r', encoding="utf-8") as file:
        return file.read()

def read_json_file(file_path):
    with open(file_path, 'r', encoding="utf-8") as file:
        return json.load(file)

def extract_raw_text(json_data):
    result = []
    if isinstance(json_data, dict):
        for key, value in json_data.items():
            if key == 'raw_text':
                result.append(value)
            else:
                result.append(extract_raw_text(value))
    elif isinstance(json_data, list):
        for item in json_data:
            result.append(extract_raw_text(item))
    return ' '.join(filter(None, result))

def clean_text(text):
    # Entferne Satzzeichen und Sonderzeichen, konvertiere in Kleinbuchstaben
    return re.sub(r'[^\w\s]', '', text).lower()

def compare_texts(text1, text2):
    words1 = sorted(clean_text(text1).split())
    words2 = sorted(clean_text(text2).split())
    return words1 == words2

def print_compare_texts(text1, text2):
    words1 = sorted(clean_text(text1).split())
    words2 = sorted(clean_text(text2).split())
    set1 = set(words1)
    set2 = set(words2)
    difference = set1.symmetric_difference(set2)

    total_words = len(set1.union(set2))
    missing_percentage = (len(difference) / total_words) * 100 if total_words > 0 else 0

    print(f"Prozent: {missing_percentage:.2f}%  Diff: {difference}")
    return len(difference) == 0, missing_percentage, difference

def move_non_matching_file(file_name, p1_nice, p1_aussortiert, folder1, folder_out):
    base_file_name = file_name.replace('.txt', '')
    for file in os.listdir(p1_nice):
        if base_file_name in file:
            src_path = os.path.join(p1_nice, file)
            dst_path = os.path.join(p1_aussortiert, file)
            shutil.move(src_path, dst_path)
            print(f"Moved {file} from {p1_nice} to {p1_aussortiert}")
            break

    ann_file = base_file_name + ".ann"
    txt_file = base_file_name + ".txt"

    ann_src_path = os.path.join(folder1, ann_file)
    txt_src_path = os.path.join(folder1, txt_file)
    ann_dst_path = os.path.join(folder_out, ann_file)
    txt_dst_path = os.path.join(folder_out, txt_file)

    if os.path.exists(ann_src_path):
        shutil.move(ann_src_path, ann_dst_path)
        print(f"Moved {ann_file} from {folder1} to {folder_out}")
    if os.path.exists(txt_src_path):
        shutil.move(txt_src_path, txt_dst_path)
        print(f"Moved {txt_file} from {folder1} to {folder_out}")

def find_non_matching_studies(folder1, folder2, p1_nice, p1_aussortiert, folder_out):
    non_matching_studies = []
    count_non_matching = 0

    for file_name in os.listdir(folder1):
        if file_name.endswith(".txt"):
            txt_file_path = os.path.join(folder1, file_name)
            json_file_name = file_name.replace('.txt', '_parsed_2.json')
            json_file_path = os.path.join(folder2, json_file_name)

            if os.path.exists(json_file_path):
                text1 = read_txt_file(txt_file_path).strip()
                json_data = read_json_file(json_file_path)
                text2 = extract_raw_text(json_data).strip()

                texts_match, missing_percentage, difference = print_compare_texts(text1, text2)

                if not texts_match and (missing_percentage > 4 or 'set' in difference or 'empty' in difference):
                    non_matching_studies.append(file_name)
                    count_non_matching += 1
                    #move_non_matching_file(file_name, p1_nice, p1_aussortiert, folder1, folder_out)

    return non_matching_studies, count_non_matching

folder1 = '../input/ann_text'
folder_out = '../input/ann_text_aus'
folder2 = 'p2'

p1_nice = "p1_label/p1_ready"
p1_aussortiert = "p1_label/p1_aussortiert"

non_matching_studies, count_non_matching = find_non_matching_studies(folder1, folder2, p1_nice, p1_aussortiert, folder_out)
print(f"Total non-matching studies: {count_non_matching}")
print("Non-matching studies:", non_matching_studies)

Prozent: 0.00%  Diff: set()
Prozent: 2.68%  Diff: {'biopsy', 'proven', 'biopsyproven'}
Prozent: 0.00%  Diff: set()
Prozent: 7.14%  Diff: {'1', '01', '0'}
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 6.12%  Diff: {'alcoholsubstance', 'substance', 'alcohol'}
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 0.00%  Diff: set()
Prozent: 2.56%  Diff: {'sulfonylureametformin'}
Prozent: 0.00%  Diff: set()
Pro

In [5]:
print(f"Anzahl der ungleichen Studien: {count_non_matching}")
print("Studiennamen mit ungleichem Inhalt:")
for study in non_matching_studies:
    print(study.split(".")[0])

Anzahl der ungleichen Studien: 94
Studiennamen mit ungleichem Inhalt:
NCT00061308_inc
NCT00182520_exc
NCT00480129_inc
NCT00576173_exc
NCT00728156_exc
NCT00917891_inc
NCT00954850_inc
NCT01218737_exc
NCT01261832_exc
NCT01491295_inc
NCT01567605_exc
NCT01579604_exc
NCT01579604_inc
NCT01581749_exc
NCT01650792_exc
NCT01669369_exc
NCT01669369_inc
NCT01715714_exc
NCT01728194_inc
NCT01822262_exc
NCT01909934_exc
NCT01943812_inc
NCT01980680_inc
NCT01994382_exc
NCT01994382_inc
NCT01996436_inc
NCT01997112_exc
NCT02105090_exc
NCT02137369_inc
NCT02162433_exc
NCT02175186_exc
NCT02186782_exc
NCT02195024_exc
NCT02224040_exc
NCT02224040_inc
NCT02321202_exc
NCT02350439_exc
NCT02357654_inc
NCT02369211_exc
NCT02384850_exc
NCT02406885_exc
NCT02413970_inc
NCT02535299_exc
NCT02550080_inc
NCT02550769_inc
NCT02552459_exc
NCT02566863_exc
NCT02573168_inc
NCT02573597_inc
NCT02595190_inc
NCT02607319_inc
NCT02630628_exc
NCT02632760_exc
NCT02637076_exc
NCT02644629_exc
NCT02691793_inc
NCT02749617_inc
NCT02785549_inc
NC

In [17]:
non_matching_studies

['NCT01217671_exc.txt',
 'NCT01793831_exc.txt',
 'NCT02224040_inc.txt',
 'NCT02357654_inc.txt',
 'NCT02566863_exc.txt',
 'NCT02573597_inc.txt',
 'NCT02893293_exc.txt',
 'NCT02944604_inc.txt',
 'NCT02966236_exc.txt',
 'NCT03034733_exc.txt',
 'NCT03431831_inc.txt',
 'NCT03495557_inc.txt']